In [2]:
import re
from PyPDF2 import PdfReader
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter

# Custom dictionary for entity recognition
custom_dictionary = {
    "Name": {
        "description": "Identifies the patient's full name or title.",
        "terms": [
            "Patient",
            "Name",
            "Patient Name",
            "Full Name",
            "Mr.",
            "Ms.",
            "Mrs.",
            "Dr."
        ]
    },
    "Age": {
        "description": "Specifies the patient’s age.",
        "terms": [
            "Age",
            "years old",
            "yrs",
            "yo"
        ]
    },
    "Date": {
        "description": "Represents the date of the report, visit, or test.",
        "terms": [
            "Date",
            "Visited on",
            "Appointment Date",
            "On"
        ]
    },
    "Symptoms": {
        "description": "Lists patient-reported symptoms and complaints.",
        "terms": [
            "cough",
            "fever",
            "fatigue",
            "shortness of breath",
            "nausea",
            "abdominal pain",
            "pain",
            "joint pain",
            "stiffness",
            "swelling",
            "headache",
            "dizziness",
            "cramping"
        ]
    },
    "Lab Values/Test Names": {
        "description": "Covers names of laboratory tests and related lab values.",
        "terms": [
            "CBC",
            "complete blood count",
            "WBC",
            "hemoglobin",
            "platelet",
            "CMP",
            "comprehensive metabolic panel",
            "electrolytes",
            "kidney function",
            "liver enzymes",
            "ALT",
            "AST",
            "lipid panel",
            "cholesterol",
            "LDL",
            "HDL",
            "triglycerides",
            "fasting blood sugar",
            "urinalysis",
            "TSH",
            "free T4",
            "X-ray",
            "ultrasound"
        ]
    },
    "Diagnosis": {
        "description": "Denotes the clinical diagnosis or impression based on the findings.",
        "terms": [
            "diagnosis",
            "diagnosed",
            "impression",
            "clinical findings",
            "appendicitis",
            "osteoporosis",
            "osteoarthritis",
            "infection",
            "inflammation",
            "metabolic syndrome"
        ]
    },
    "Treatment Medications": {
        "description": "Lists prescribed treatments, medications, and management strategies.",
        "terms": [
            "prescribed",
            "treatment",
            "antibiotics",
            "bronchodilator",
            "NSAIDs",
            "pain reliever",
            "physiotherapy",
            "lifestyle modifications",
            "diet",
            "exercise",
            "over-the-counter",
            "medications",
            "non-steroidal anti-inflammatory drugs"
        ]
    }
}

def extract_text_from_pdf(file_path):
    """Extracts text from a PDF file using PyPDF2."""
    text = ""
    with open(file_path, "rb") as f:
        pdf_reader = PdfReader(f)
        for page in pdf_reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

def extract_entities(text, dictionary):
    """
    Extracts entities from text based on the provided dictionary.
    For each category, it searches for the defined terms and collects matches.
    """
    extracted_data = {}
    for category, info in dictionary.items():
        extracted_data[category] = []
        for term in info["terms"]:
            # Use regex to match whole words (case insensitive)
            pattern = r'\b' + re.escape(term) + r'\b'
            matches = re.findall(pattern, text, flags=re.IGNORECASE)
            if matches:
                extracted_data[category].extend(matches)
        # Remove duplicates and sort the results
        extracted_data[category] = sorted(list(set(extracted_data[category])))
    return extracted_data

def create_structured_pdf(data, output_path):
    """
    Creates a structured PDF with the extracted entities.
    Each entity category is shown as a heading followed by a list of found entities.
    """
    c = canvas.Canvas(output_path, pagesize=letter)
    width, height = letter
    margin = 50
    y = height - margin

    # Title
    c.setFont("Helvetica-Bold", 16)
    c.drawString(margin, y, "Structured Medical Report")
    y -= 30

    # Write each entity category and its items
    for category, items in data.items():
        c.setFont("Helvetica-Bold", 14)
        c.drawString(margin, y, category + ":")
        y -= 20
        c.setFont("Helvetica", 12)
        if items:
            for item in items:
                c.drawString(margin + 20, y, "- " + item)
                y -= 15
                # Create new page if needed
                if y < margin:
                    c.showPage()
                    y = height - margin
        else:
            c.drawString(margin + 20, y, "No entities found.")
            y -= 15
            if y < margin:
                c.showPage()
                y = height - margin
        y -= 10
        if y < margin:
            c.showPage()
            y = height - margin

    c.save()

# Main execution
if __name__ == "__main__":
    input_pdf = "report1.pdf"  # Replace with your PDF file path
    output_pdf = "structured_output.pdf"  # Desired output PDF file path

    # Extract text from the PDF
    print("Extracting text from PDF...")
    extracted_text = extract_text_from_pdf(input_pdf)

    # Run entity extraction using our custom dictionary
    print("Extracting entities...")
    extracted_entities = extract_entities(extracted_text, custom_dictionary)

    # Generate a structured PDF output
    print("Creating structured PDF...")
    create_structured_pdf(extracted_entities, output_pdf)

    print("Structured PDF created:", output_pdf)


Extracting text from PDF...
Extracting entities...
Creating structured PDF...
Structured PDF created: structured_output.pdf
